<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

# ¿Qué es un decorador?
**Taller de Programación para el Análisis de Datos — FCE-UBA**

Un **decorador** es una función que **le agrega algo a otra función**, sin modificar su código.

Se escribe con `@` arriba de la función:

```python
@mi_decorador
def mi_funcion():
    ...
```

Para entender cómo funciona, primero necesitamos dos ideas previas.

---
#### 1. Idea previa: una función se puede guardar en una variable

En Python, una función es un valor más, como un número o un texto. Se puede guardar en una variable y llamarla con otro nombre:

In [ ]:
def saludar():
    print("Hola!")

otra = saludar    # sin paréntesis: NO la ejecuta, la guarda
otra()            # con paréntesis: la ejecuta

Hola!


> **Clave:** `saludar` (sin paréntesis) es la función. `saludar()` (con paréntesis) es ejecutarla.

---
#### 2. Idea previa: una función puede recibir otra función

Como una función es un valor, se la podemos pasar a otra función como parámetro:

In [ ]:
def ejecutar_dos_veces(funcion):
    funcion()
    funcion()

ejecutar_dos_veces(saludar)

Hola!
Hola!


---
#### 3. Armamos un decorador a mano

Queremos que, cada vez que se ejecute `saludar`, aparezca un mensaje **antes** y otro **después**, sin tocar el código de `saludar`.

Escribimos una función `avisar` que:
1. recibe una función,
2. arma una función **nueva** que hace "algo antes + la función original + algo después",
3. devuelve esa función nueva.

In [ ]:
def avisar(funcion):
    def nueva_funcion():
        print(">>> Empieza")
        funcion()            # acá se ejecuta la función original
        print(">>> Termina")
    return nueva_funcion     # devolvemos la función nueva (sin ejecutarla)

In [ ]:
def saludar():
    print("Hola!")

saludar = avisar(saludar)   # reemplazamos saludar por la versión "decorada"

saludar()

>>> Empieza
Hola!
>>> Termina


Eso es un decorador: `avisar` **envolvió** a `saludar` y le agregó los mensajes.

---
#### 4. La forma corta: `@`

Escribir `saludar = avisar(saludar)` es medio incómodo. Python tiene un atajo: poner `@avisar` arriba de la función.

Estas dos formas son **exactamente lo mismo**:

| Forma larga | Forma corta |
|---|---|
| `def saludar():`<br>&nbsp;&nbsp;&nbsp;&nbsp;`print("Hola!")`<br>`saludar = avisar(saludar)` | `@avisar`<br>`def saludar():`<br>&nbsp;&nbsp;&nbsp;&nbsp;`print("Hola!")` |

In [ ]:
@avisar
def saludar():
    print("Hola!")

saludar()

>>> Empieza
Hola!
>>> Termina


El mismo decorador se puede usar con cualquier función:

In [ ]:
@avisar
def despedir():
    print("Chau!")

despedir()

>>> Empieza
Chau!
>>> Termina


---
#### 5. Funciones con parámetros

Si la función decorada recibe parámetros, la función nueva también tiene que recibirlos y pasárselos a la original:

In [ ]:
def avisar(funcion):
    def nueva_funcion(nombre):
        print(">>> Empieza")
        funcion(nombre)
        print(">>> Termina")
    return nueva_funcion

@avisar
def saludar(nombre):
    print(f"Hola, {nombre}!")

saludar("Ana")

>>> Empieza
Hola, Ana!
>>> Termina


Pero así el decorador solo sirve para funciones con **un** parámetro. Para que sirva con cualquier cantidad, se usan `*args` y `**kwargs`, que significan "todos los parámetros que lleguen, sean los que sean":

In [ ]:
def avisar(funcion):
    def nueva_funcion(*args, **kwargs):
        print(">>> Empieza")
        resultado = funcion(*args, **kwargs)   # pasamos todo tal cual
        print(">>> Termina")
        return resultado                       # devolvemos lo que devolvió la original
    return nueva_funcion

@avisar
def sumar(a, b):
    return a + b

total = sumar(3, 5)
print("Resultado:", total)

>>> Empieza
>>> Termina
Resultado: 8


> Importante: si la función original devuelve algo (como `sumar`), la función nueva tiene que hacer `return resultado`. Si no, el valor se pierde y se obtiene `None`.

---
#### 6. Un ejemplo útil: medir cuánto tarda una función

Un uso clásico de los decoradores es medir el tiempo de ejecución. El decorador mira el reloj antes y después:

In [ ]:
import time

def medir_tiempo(funcion):
    def nueva_funcion(*args, **kwargs):
        inicio = time.time()
        resultado = funcion(*args, **kwargs)
        fin = time.time()
        print(f"{funcion.__name__} tardó {fin - inicio:.4f} segundos")
        return resultado
    return nueva_funcion

In [ ]:
@medir_tiempo
def sumar_hasta(n):
    total = 0
    for i in range(n):
        total += i
    return total

print(sumar_hasta(1_000_000))
print(sumar_hasta(5_000_000))

sumar_hasta tardó 0.0348 segundos
499999500000


sumar_hasta tardó 0.1562 segundos
12499997500000


`funcion.__name__` es el nombre de la función original; lo usamos para que el mensaje diga cuál se midió.

---
#### 7. Decoradores que ya existen

Muchas librerías traen decoradores listos para usar. Por ejemplo:

| Decorador | ¿Qué agrega? |
|---|---|
| `@validate_call` (pydantic) | Controla que los parámetros sean del tipo correcto |
| `@property` (Python) | Permite usar un método como si fuera un atributo |
| `@lru_cache` (Python, módulo `functools`) | Recuerda resultados ya calculados para no repetir el cálculo |

Un ejemplo con `@lru_cache`: la segunda vez que pedimos el mismo valor, la respuesta es instantánea porque ya estaba guardada.

In [ ]:
from functools import lru_cache

@medir_tiempo
@lru_cache
def calculo_lento(n):
    time.sleep(1)   # simulamos un cálculo que tarda 1 segundo
    return n * 2

print(calculo_lento(10))   # tarda 1 segundo
print(calculo_lento(10))   # instantáneo: ya lo tenía guardado

calculo_lento tardó 1.0003 segundos
20
calculo_lento tardó 0.0000 segundos
20


> Se pueden apilar varios decoradores. Se aplican de abajo hacia arriba: primero `@lru_cache` y después `@medir_tiempo`.

---
#### 9. Ejercicios

1. Escribí un decorador `repetir_tres` que haga que la función decorada se ejecute tres veces. Probalo con una función que imprima `"Hola"`.

2. Escribí un decorador `mostrar_parametros` que, antes de ejecutar la función, imprima los parámetros que recibió. Probalo con `sumar(3, 5)`.

3. Escribí un decorador `en_mayusculas` para funciones que devuelven un texto, que devuelva ese texto en mayúsculas. Probalo con una función `saludo(nombre)` que devuelva `f"hola {nombre}"`.